# Preparação e análise descritiva final — DRC no SUS

Notebook dedicado a transformar as fontes agregadas já existentes em bases tratadas, executar validações e produzir sumários/gráficos descritivos para 2024–2025. O nível analítico é UF-mês; nenhuma pessoa é vinculada entre sistemas.

**Escopo:** produção aprovada de hemodiálise no SIA/SUS, internações N17–N19 do TabNet SIH/SUS e população IBGE por UF-ano. **N18 permanece preliminar e excluído dos totais finais.** APAC e DBCs não são relidos aqui: suas pendências de semântica/identidade e a divergência de N18 pertencem aos notebooks de auditoria. Este notebook não estima eficácia ou efetividade.

In [1]:
from pathlib import Path
import csv
import re
import math
from collections import defaultdict
from datetime import date
from html import escape

candidates = [Path.cwd() / 'Trabalho-final', Path.cwd(), Path.cwd().parent]
project_dir = next(p.resolve() for p in candidates if (p / 'Dados').is_dir())
data_dir = project_dir / 'Dados'
source_dir = data_dir / '02_derivados_preliminares'
out_dir = data_dir / '03_bases_analiticas/descritiva_territorial_2024_2025'
figure_dir = out_dir / 'graficos'
out_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

sia_path = source_dir / 'SIA/hemodialise_sia_sus_uf_mes_2024_2025.csv'
sih_path = source_dir / 'SIH/sih_insuficiencia_renal_n17_n19_uf_mes_2024_2025_analitico.csv'
pop_path = source_dir / 'IBGE/populacao_ibge_uf_2024_2025.csv'
n18_path = source_dir / 'SIH/sih_n18_microdados_uf_ano_mes_2024_2025.csv'
assert all(p.exists() for p in [sia_path, sih_path, pop_path, n18_path]), 'Fonte local ausente: não baixar novamente sem verificar inventário.'

def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f, delimiter=';'))
def write_csv(path, rows):
    if not rows:
        raise ValueError(f'Não há linhas para gravar: {path.name}')
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]), delimiter=';', extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)
print('Projeto:', project_dir)
print('Saídas tratadas:', out_dir)
print('Fontes existentes; dados brutos e CSVs de origem permanecem intocados.')

Projeto: /Users/paulis/Documents/Projects/POS-SENAI/estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final
Saídas tratadas: /Users/paulis/Documents/Projects/POS-SENAI/estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados/03_bases_analiticas/descritiva_territorial_2024_2025
Fontes existentes; dados brutos e CSVs de origem permanecem intocados.


## 1. Transformação e validação das fontes

Converte SIA e TabNet de matriz UF × mês para formato longo; organiza população em dimensão UF-ano; junta por códigos oficiais de UF, ano e mês. Linhas administrativas de total são descartadas. Valores SIH não numéricos/vazios ficam ausentes, nunca são substituídos por zero. As checagens abaixo verificam cobertura, unicidade das chaves, totais e ausência conhecida de RR em dezembro de 2025.

In [2]:
meses = {'Jan':1,'Fev':2,'Mar':3,'Abr':4,'Mai':5,'Jun':6,'Jul':7,'Ago':8,'Set':9,'Out':10,'Nov':11,'Dez':12}
competencia = re.compile(r'(2024|2025)/([A-Za-zÀ-ÿ]+)')
# SIA
sia_source = read_csv(sia_path)
sia = {}
for row in sia_source:
    match_uf = re.match(r'\s*(\d{2})', row['UF'])
    if not match_uf and row['UF'].strip().lower().startswith('total'):
        continue
    assert match_uf, f'UF não reconhecida em SIA: {row["UF"]}'
    cod = match_uf.group(1)
    for col, raw in row.items():
        m = competencia.fullmatch(col)
        if m and m.group(2) in meses:
            key = (cod, int(m.group(1)), meses[m.group(2)])
            assert key not in sia, f'Chave SIA duplicada: {key}'
            sia[key] = int(raw)
assert len(sia) == 648, f'Esperadas 648 células SIA; encontradas {len(sia)}'

# SIH TabNet
sih_source = read_csv(sih_path)
sih = {}
for row in sih_source:
    raw_code = (row.get('Código UF') or '').strip()
    if not raw_code.isdigit():
        continue
    cod = raw_code.zfill(2)
    for col, raw in row.items():
        m = competencia.fullmatch(col)
        if m and m.group(2) in meses:
            key = (cod, int(m.group(1)), meses[m.group(2)])
            assert key not in sih, f'Chave SIH duplicada: {key}'
            val = (raw or '').strip()
            sih[key] = int(val) if val.isdigit() else None
assert len(sih) == 648, f'Esperadas 648 células SIH; encontradas {len(sih)}'

# IBGE UF-ano
pop_source = read_csv(pop_path)
pop = {}
for row in pop_source:
    key = (row['cod_uf'].zfill(2), int(row['ano']))
    assert key not in pop, f'Chave IBGE duplicada: {key}'
    pop[key] = {'regiao': row['regiao'], 'sigla_uf': row['sigla_uf'], 'uf': row['uf'],
                'populacao_estimada': int(row['populacao_estimada']),
                'data_referencia': row['data_referencia']}
assert len(pop) == 54, f'Esperadas 54 células UF-ano de população; encontradas {len(pop)}'
assert len({k[0] for k in sia}) == 27 and len({k[1:] for k in sia}) == 24

painel = []
for key in sorted(sia):
    cod, ano, mes = key
    p = pop[(cod, ano)]
    val_sih = sih[key]
    painel.append({'cod_uf':cod, 'sigla_uf':p['sigla_uf'], 'uf':p['uf'], 'regiao':p['regiao'],
                   'ano':ano, 'mes':mes, 'competencia':f'{ano}-{mes:02d}',
                   'sia_hemodialise_quantidade_aprovada':sia[key],
                   'sih_n17_n19_internacoes_tabnet':val_sih,
                   'sih_status':'observado' if val_sih is not None else 'ausente_ou_suprimido',
                   'populacao_ibge_referencia_anual':p['populacao_estimada'],
                   'populacao_data_referencia':p['data_referencia']})
assert len(painel) == 648
assert len({(r['cod_uf'],r['ano'],r['mes']) for r in painel}) == 648
faltantes = [(r['sigla_uf'],r['competencia']) for r in painel if r['sih_status'] != 'observado']
assert faltantes == [('RR','2025-12')], f'Rever células SIH ausentes: {faltantes}'
assert sum(r['sih_n17_n19_internacoes_tabnet'] or 0 for r in painel) == 309574
assert sum(r['sia_hemodialise_quantidade_aprovada'] for r in painel) > 0

write_csv(out_dir/'base_descritiva_sia_hemodialise_uf_mes.csv', [
    {'cod_uf':k[0], 'ano':k[1], 'mes':k[2], 'competencia':f'{k[1]}-{k[2]:02d}',
     'sia_hemodialise_quantidade_aprovada':v} for k,v in sorted(sia.items())])
write_csv(out_dir/'base_descritiva_sih_n17_n19_uf_mes.csv', [
    {'cod_uf':k[0], 'ano':k[1], 'mes':k[2], 'competencia':f'{k[1]}-{k[2]:02d}',
     'sih_n17_n19_internacoes_tabnet':v,
     'status':'observado' if v is not None else 'ausente_ou_suprimido'} for k,v in sorted(sih.items())])
write_csv(out_dir/'dimensao_populacao_ibge_uf_ano.csv', [
    {'cod_uf':k[0], 'ano':k[1], **v} for k,v in sorted(pop.items())])
write_csv(out_dir/'painel_descritivo_uf_mes_2024_2025.csv', painel)
print(f'Validação estrutural: {len(painel)} linhas UF-mês; {len(pop)} linhas UF-ano; {len(faltantes)} célula SIH ausente: {faltantes[0]}.')
print('Soma SIH TabNet observada:', sum(r['sih_n17_n19_internacoes_tabnet'] or 0 for r in painel))
print('N18 preliminar foi excluído do painel e não participa de totalizações.')

Validação estrutural: 648 linhas UF-mês; 54 linhas UF-ano; 1 célula SIH ausente: ('RR', '2025-12').
Soma SIH TabNet observada: 309574
N18 preliminar foi excluído do painel e não participa de totalizações.


## 2. Sumários por UF, região e ano

Os totais anuais SIH são rotulados como **observados** e acompanhados da quantidade de meses disponíveis; o ano de 2025 não será apresentado como completo para RR. Média mensal é apenas média dos meses observados. A produção SIA continua sendo quantidade aprovada, não pacientes únicos. Não serão calculadas taxas mensais com a população de julho.

In [3]:
por_uf_ano = defaultdict(lambda: {'sia_total':0, 'sih_total_observado':0, 'sih_meses_observados':0, 'sih_meses_ausentes':0})
por_regiao_ano_mes = defaultdict(lambda: {'sia':0, 'sih_observado':0, 'sih_ufs_observadas':0, 'sih_ufs_esperadas':0})
por_brasil_mes = defaultdict(lambda: {'sia':0, 'sih_observado':0, 'sih_ufs_observadas':0, 'sih_ufs_esperadas':27})
for r in painel:
    a = por_uf_ano[(r['cod_uf'],r['ano'])]
    a['sia_total'] += r['sia_hemodialise_quantidade_aprovada']
    if r['sih_n17_n19_internacoes_tabnet'] is None:
        a['sih_meses_ausentes'] += 1
    else:
        a['sih_total_observado'] += r['sih_n17_n19_internacoes_tabnet']
        a['sih_meses_observados'] += 1
    reg = por_regiao_ano_mes[(r['regiao'],r['ano'],r['mes'])]
    reg['sih_ufs_esperadas'] = len({x['cod_uf'] for x in painel if x['regiao']==r['regiao'] and x['ano']==r['ano'] and x['mes']==r['mes']})
    reg['sia'] += r['sia_hemodialise_quantidade_aprovada']
    if r['sih_n17_n19_internacoes_tabnet'] is not None:
        reg['sih_observado'] += r['sih_n17_n19_internacoes_tabnet']; reg['sih_ufs_observadas'] += 1
    b = por_brasil_mes[(r['ano'],r['mes'])]
    b['sia'] += r['sia_hemodialise_quantidade_aprovada']
    if r['sih_n17_n19_internacoes_tabnet'] is not None:
        b['sih_observado'] += r['sih_n17_n19_internacoes_tabnet']; b['sih_ufs_observadas'] += 1

resumo_uf_ano=[]
for (cod,ano),v in sorted(por_uf_ano.items()):
    p=pop[(cod,ano)]
    resumo_uf_ano.append({'cod_uf':cod,'sigla_uf':p['sigla_uf'],'uf':p['uf'],'regiao':p['regiao'],'ano':ano,
      'sia_total_quantidade_aprovada':v['sia_total'],'sih_total_observado_n17_n19':v['sih_total_observado'],
      'sih_meses_observados':v['sih_meses_observados'],'sih_meses_ausentes':v['sih_meses_ausentes'],
      'sih_media_mensal_meses_observados':round(v['sih_total_observado']/v['sih_meses_observados'],2) if v['sih_meses_observados'] else '',
      'populacao_ibge_referencia_anual':p['populacao_estimada'],'populacao_data_referencia':p['data_referencia']})
serie_brasil=[]
for (ano,mes),v in sorted(por_brasil_mes.items()):
    serie_brasil.append({'ano':ano,'mes':mes,'competencia':f'{ano}-{mes:02d}',
      'sia_total_quantidade_aprovada':v['sia'],'sih_total_observado_n17_n19':v['sih_observado'],
      'sih_ufs_observadas':v['sih_ufs_observadas'],'sih_ufs_esperadas':v['sih_ufs_esperadas']})
serie_regional=[]
for (reg,ano,mes),v in sorted(por_regiao_ano_mes.items()):
    serie_regional.append({'regiao':reg,'ano':ano,'mes':mes,'competencia':f'{ano}-{mes:02d}',
       'sia_total_quantidade_aprovada':v['sia'],'sih_total_observado_n17_n19':v['sih_observado'],
       'sih_ufs_observadas':v['sih_ufs_observadas'],'sih_ufs_esperadas':v['sih_ufs_esperadas']})
write_csv(out_dir/'resumo_descritivo_uf_ano.csv',resumo_uf_ano)
write_csv(out_dir/'serie_mensal_brasil.csv',serie_brasil)
write_csv(out_dir/'serie_mensal_regiao.csv',serie_regional)
print('Estados-ano:',len(resumo_uf_ano),'| Brasil-mês:',len(serie_brasil),'| Região-mês:',len(serie_regional))
print('Totais Brasil por ano (SIH observado; cobertura de células):')
for ano in (2024,2025):
    rr=[x for x in serie_brasil if x['ano']==ano]
    print(ano, 'SIA=',sum(x['sia_total_quantidade_aprovada'] for x in rr),
          'SIH observado=',sum(x['sih_total_observado_n17_n19'] for x in rr),
          'meses SIH com cobertura completa de UFs=',sum(x['sih_ufs_observadas']==27 for x in rr),'de',len(rr))

Estados-ano: 54 | Brasil-mês: 24 | Região-mês: 120
Totais Brasil por ano (SIH observado; cobertura de células):
2024 SIA= 18311982 SIH observado= 156491 meses SIH com cobertura completa de UFs= 12 de 12
2025 SIA= 18984870 SIH observado= 153083 meses SIH com cobertura completa de UFs= 11 de 12


## 2A. Comparação de períodos equivalentes (janeiro–novembro)

Esta comparação janeiro–novembro é mantida como saída descritiva de referência. A exploração posterior identificou queda ampla no SIH em novembro/dezembro de 2025; por isso, as hipóteses principais passaram a usar janeiro–outubro enquanto se verifica a maturidade da fonte. São mudanças descritivas de registros administrativos, não variações de pacientes, risco ou efeito do tratamento.

In [4]:
comparacao_uf=[]
for cod in sorted({r['cod_uf'] for r in painel}):
    p=pop[(cod,2024)]
    subset={(r['ano'],r['mes']):r for r in painel if r['cod_uf']==cod}
    linha={'cod_uf':cod,'sigla_uf':p['sigla_uf'],'uf':p['uf'],'regiao':p['regiao'],
           'periodo_comparado':'janeiro-novembro'}
    for nome, campo in [('sia','sia_hemodialise_quantidade_aprovada'),('sih','sih_n17_n19_internacoes_tabnet')]:
        v24=[subset[(2024,m)][campo] for m in range(1,12)]
        v25=[subset[(2025,m)][campo] for m in range(1,12)]
        assert all(v is not None for v in v24+v25), f'Período não completo para {cod}/{nome}'
        t24=sum(v24); t25=sum(v25)
        linha[f'{nome}_jan_nov_2024']=t24
        linha[f'{nome}_jan_nov_2025']=t25
        linha[f'{nome}_diferenca_absoluta']=t25-t24
        linha[f'{nome}_variacao_percentual']=round((t25/t24-1)*100,2) if t24 else ''
    comparacao_uf.append(linha)
assert len(comparacao_uf)==27
write_csv(out_dir/'comparacao_periodos_equivalentes_uf.csv',comparacao_uf)

comparacao_brasil=[]
for nome in ('sia','sih'):
    t24=sum(r[f'{nome}_jan_nov_2024'] for r in comparacao_uf)
    t25=sum(r[f'{nome}_jan_nov_2025'] for r in comparacao_uf)
    comparacao_brasil.append({'indicador':nome,'periodo':'janeiro-novembro','ufs_comparaveis':27,
      'total_2024':t24,'total_2025':t25,'diferenca_absoluta':t25-t24,
      'variacao_percentual':round((t25/t24-1)*100,2) if t24 else ''})
write_csv(out_dir/'comparacao_periodos_equivalentes_brasil.csv',comparacao_brasil)
print('Comparação comum janeiro–novembro (27 UFs):')
for r in comparacao_brasil:
    print(r['indicador'],r['total_2024'],'→',r['total_2025'],
          '| diferença',r['diferenca_absoluta'],'| variação %',r['variacao_percentual'])

Comparação comum janeiro–novembro (27 UFs):
sia 16733358 → 17310415 | diferença 577057 | variação % 3.45
sih 143806 → 146581 | diferença 2775 | variação % 1.93


## 3. Gráficos descritivos

Os gráficos são SVG gerados apenas com a biblioteca padrão do Python, sem instalar dependências. A série SIH é soma das células observadas; a cobertura de UFs aparece no gráfico e a lacuna não é interpolada. As escalas dos dois painéis são independentes, pois SIA e SIH medem fenômenos diferentes.

In [5]:
def svg_line_chart(path, title, series, y_label, colors, coverage=None):
    # series: lista de (rótulo, valores); valores alinhados a 24 competências.
    W,H=1100,560; left,right,top,bottom=90,30,75,90
    plot_w,plot_h=W-left-right,H-top-bottom
    all_values=[v for _,vals in series for v in vals if v is not None]
    ymax=max(all_values) if all_values else 1
    ymax=math.ceil(ymax/5)*5 if ymax<50 else math.ceil(ymax/100)*100
    x=lambda i: left+i*plot_w/23
    y=lambda v: top+plot_h-(v/ymax)*plot_h
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}">',
      '<rect width="100%" height="100%" fill="white"/>',
      f'<text x="{left}" y="36" font-family="Arial" font-size="24" font-weight="bold">{escape(title)}</text>',
      f'<text transform="translate(24 {top+plot_h/2}) rotate(-90)" text-anchor="middle" font-family="Arial" font-size="14">{escape(y_label)}</text>']
    for tick in range(6):
        val=ymax*tick/5; yy=y(val)
        parts.append(f'<line x1="{left}" y1="{yy:.1f}" x2="{W-right}" y2="{yy:.1f}" stroke="#dddddd"/>')
        parts.append(f'<text x="{left-10}" y="{yy+5:.1f}" text-anchor="end" font-family="Arial" font-size="12">{val:,.0f}</text>')
    for i in range(0,24,2):
        parts.append(f'<text x="{x(i):.1f}" y="{H-55}" text-anchor="middle" font-family="Arial" font-size="11">{2024+i//12}-{i%12+1:02d}</text>')
    for j,(label,vals) in enumerate(series):
        seg=[]
        for i,val in enumerate(vals):
            if val is None:
                if len(seg)>1: parts.append(f'<polyline fill="none" stroke="{colors[j]}" stroke-width="2.5" points="{" ".join(seg)}"/>')
                seg=[]
            else: seg.append(f'{x(i):.1f},{y(val):.1f}')
        if len(seg)>1: parts.append(f'<polyline fill="none" stroke="{colors[j]}" stroke-width="2.5" points="{" ".join(seg)}"/>')
        ly=top+j*23
        parts.append(f'<line x1="{left+10}" y1="{ly}" x2="{left+35}" y2="{ly}" stroke="{colors[j]}" stroke-width="3"/>')
        parts.append(f'<text x="{left+42}" y="{ly+5}" font-family="Arial" font-size="13">{escape(label)}</text>')
    if coverage is not None:
        cov='Cobertura SIH: '+', '.join(f'{v}/27 UFs' for v in coverage)
        parts.append(f'<text x="{W-right}" y="{H-18}" text-anchor="end" font-family="Arial" font-size="12" fill="#555">{escape(cov)}</text>')
    parts.append('</svg>')
    path.write_text('\n'.join(parts),encoding='utf-8')

months=sorted((ano,mes) for ano in (2024,2025) for mes in range(1,13))
series_brasil=[('SIA — quantidade aprovada',[por_brasil_mes[k]['sia'] for k in months]),
               ('SIH N17–N19 — total observado',[por_brasil_mes[k]['sih_observado'] for k in months])]
svg_line_chart(figure_dir/'serie_mensal_brasil.svg','SUS — produção SIA e internações SIH observadas',series_brasil,
               'Contagem administrativa (escala própria por série)',['#1261a0','#b33b2e'],
               [por_brasil_mes[k]['sih_ufs_observadas'] for k in months])
cores=['#1f77b4','#ff7f0e','#2ca02c','#9467bd','#17becf']
regioes=['Norte','Nordeste','Sudeste','Sul','Centro-Oeste']
series_reg=[]
for reg in regioes:
    series_reg.append((reg,[por_regiao_ano_mes[(reg,a,m)]['sia'] for a,m in months]))
svg_line_chart(figure_dir/'serie_mensal_sia_por_regiao.svg','SIA — quantidade aprovada de hemodiálise por região',series_reg,
               'Quantidade aprovada',cores)
series_reg_sih=[]
for reg in regioes:
    vals=[]
    for a,m in months:
        v=por_regiao_ano_mes[(reg,a,m)]
        vals.append(v['sih_observado'] if v['sih_ufs_observadas']==v['sih_ufs_esperadas'] else None)
series_reg_sih.append((reg,vals))
svg_line_chart(figure_dir/'serie_mensal_sih_por_regiao.svg','SIH N17–N19 — internações por região (meses completos)',series_reg_sih,
               'Internações observadas',cores)
print('Gráficos SVG gerados:',*[p.name for p in sorted(figure_dir.glob('*.svg'))],sep='\n- ')

Gráficos SVG gerados:
- serie_mensal_brasil.svg
- serie_mensal_sia_por_regiao.svg
- serie_mensal_sih_por_regiao.svg


## 4. Resultado, interpretação e pendências

**Resultados da comparação janeiro–novembro (27 UFs):** SIA totalizou 16.733.358 em 2024 e 17.310.415 em 2025 (+577.057; +3,45%). SIH TabNet N17–N19 totalizou 143.806 e 146.581 (+2.775; +1,93%). Por UF, SIA aumentou em 24 e caiu em 3 UFs (−2,12% no DF a +20,49% no TO); SIH aumentou em 14 e caiu em 13 (−35,23% em RR a +37,46% no AP). São variações descritivas de registros administrativos, sem ajuste de confundimento.

A soma SIH observada nas 647 células disponíveis é 309.574; RR/dezembro/2025 está ausente/suprimida e não foi imputada. A comparação anual usa janeiro–novembro. A síntese interpretativa está em [`resultados_descritivos_territoriais_2024_2025.md`](../resultados_descritivos_territoriais_2024_2025.md).

- O produto principal é um painel descritivo agregado UF-mês; cada linha não representa um paciente.
- Produção SIA e internações SIH não têm unidade intercambiável e não devem ser divididas entre si para obter cobertura ou eficácia.
- N17–N19 é uma categoria diagnóstica ampla; não equivale a internações de pessoas em hemodiálise.
- RR/dezembro de 2025 permanece ausente/suprimida. Totais SIH 2025 são observados, não completos para todas as UFs; gráficos regionais SIH deixam lacunas onde a cobertura regional não é completa.
- População IBGE está disponível por ano com referência em 1º de julho. Não foi replicada como denominador mensal nem usada para taxa neste notebook.
- N18 continua preliminar devido à divergência com TabNet; não foi integrado nem usado como total final.
- Não há ligação individual, comparador clínico definido, tempo zero, seguimento validado ou desfecho longitudinal suficiente para avaliar efetividade.

**Pendências seguintes:** discutir a pergunta e o comparador com nefrologista/epidemiologista clínico; validar regras de datas e continuidade APAC e estabilidade dos identificadores; identificar fonte adequada de elegíveis e desfechos; manter em paralelo a reconciliação N18/TabNet. Atualizar este notebook apenas quando houver regra/documentação nova. Não interpretar tendências ecológicas como efeito do tratamento.

## 5. Exploração associativa e testes de hipóteses (exploratórios)

Esta etapa formula perguntas sobre padrões entre UFs, não sobre resultados individuais. A unidade continua sendo UF; não se vincula SIA a pessoas ou APAC. Para os testes principais, usa-se janeiro-outubro de ambos os anos, pois novembro/dezembro SIH mostram quedas amplas que podem refletir incompletude. N18 permanece excluído.

**Hipóteses:** (H1) em quantas UFs a produção SIA de janeiro–novembro mudou na mesma direção entre 2024 e 2025? (H2) a direção das internações SIH N17–N19 mudou em mais ou menos da metade das UFs? (H3) entre UFs, quantidade aprovada SIA por 100 mil habitantes se associa monotonicamente a internações N17–N19 por 100 mil habitantes em cada ano? (H4) as mudanças percentuais SIA e SIH entre anos se associam entre UFs?

H1–H2 usam teste binomial exato de sinais pareados entre UFs. H3–H4 usam Spearman com p-valor por permutação Monte Carlo (9.999 permutações, semente fixa); p-valores são ajustados pelo método de Holm dentro desta família de cinco testes. As taxas usam população IBGE anual de referência em 1º de julho e numeradores de janeiro–novembro; são indicadores ecológicos brutos, não riscos individuais nem taxas pessoa-tempo. A dependência espacial entre UFs e fatores de confundimento não são modelados, então os testes são exploratórios e não devem fundamentar afirmações causais.


In [6]:
# Exploração ecológica reproduzível, sem dependências além da biblioteca padrão.
import random
from math import comb, sqrt
from statistics import median
from collections import defaultdict

def read_analysis_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        sample = f.read(4096)
        f.seek(0)
        dialect = csv.Sniffer().sniff(sample, delimiters=',;')
        return list(csv.DictReader(f, dialect=dialect))

panel_rows = read_analysis_csv(out_dir / 'painel_descritivo_uf_mes_2024_2025.csv')
assert len(panel_rows) == 648
# Nov/Dec SIH show broad declines; use Jan-Oct as the primary comparable window
# for hypothesis tests pending confirmation of release maturity.
period_months = range(1,11)
uf_rows = {}
for row in panel_rows:
    uf= row['sigla_uf']; year=int(row['ano']); month=int(row['mes'])
    uf_rows[(uf,year,month)]=row
cmp_uf=[]
for uf in sorted({r['sigla_uf'] for r in panel_rows}):
    first=next(r for r in panel_rows if r['sigla_uf']==uf and int(r['ano'])==2024)
    entry={'cod_uf':first['cod_uf'],'sigla_uf':uf,'uf':first['uf'],'regiao':first['regiao'],'periodo':'janeiro-outubro'}
    for metric,col in [('sia','sia_hemodialise_quantidade_aprovada'),('sih','sih_n17_n19_internacoes_tabnet')]:
        vals={year:[] for year in (2024,2025)}
        for year in (2024,2025):
            for month in period_months:
                value=uf_rows[(uf,year,month)][col]
                assert value not in ('',None), f'Valor ausente inesperado: {uf}/{year}/{month}/{metric}'
                vals[year].append(int(value))
        t24=sum(vals[2024]);t25=sum(vals[2025])
        entry[f'{metric}_jan_out_2024']=t24
        entry[f'{metric}_jan_out_2025']=t25
        entry[f'{metric}_diferenca']=t25-t24
        entry[f'{metric}_variacao_percentual']=round((t25/t24-1)*100,2) if t24 else ''
    cmp_uf.append(entry)
assert len(cmp_uf)==27
write_csv(out_dir/'comparacao_periodos_jan_outubro_uf.csv',cmp_uf)

# Frequências direcionais, globais e por região.
frequency_rows=[]
for metric,field in [('SIA','sia_variacao_percentual'),('SIH_N17_N19','sih_variacao_percentual')]:
    groups=defaultdict(list)
    for row in cmp_uf:
        change=float(row[field])
        direction='aumentou' if change>0 else ('diminuiu' if change<0 else 'sem_variacao')
        groups[('Brasil',direction)].append(row['sigla_uf'])
        groups[(row['regiao'],direction)].append(row['sigla_uf'])
    for (region,direction),ufs in sorted(groups.items()):
        frequency_rows.append({'indicador':metric,'abrangencia':region,'direcao':direction,
                               'n_ufs':len(ufs),'ufs':','.join(sorted(ufs))})
write_csv(out_dir/'frequencias_direcao_variacao_uf.csv',frequency_rows)

# Teste binomial bilateral exato de sinais; empates são excluídos.
def sign_test(values):
    pos=sum(v>0 for v in values);neg=sum(v<0 for v in values);n=pos+neg;k=min(pos,neg)
    p=min(1.0,2*sum(comb(n,j) for j in range(k+1))/(2**n)) if n else 1.0
    return {'n':n,'positivos':pos,'negativos':neg,'empates':len(values)-n,
            'mediana_diferenca':median(values),'p_bruto':p}

# Ranks médios para empates; Spearman é Pearson aplicado aos ranks.
def average_ranks(values):
    order=sorted(range(len(values)),key=lambda i:values[i]);ranks=[0.0]*len(values);i=0
    while i<len(order):
        j=i+1
        while j<len(order) and values[order[j]]==values[order[i]]:j+=1
        rank=(i+1+j)/2
        for k in range(i,j):ranks[order[k]]=rank
        i=j
    return ranks

def pearson(x,y):
    mx=sum(x)/len(x);my=sum(y)/len(y);dx=[v-mx for v in x];dy=[v-my for v in y]
    denom=sqrt(sum(v*v for v in dx)*sum(v*v for v in dy))
    return sum(a*b for a,b in zip(dx,dy))/denom if denom else 0.0

def spearman_permutation(x,y,seed,permutations=9999):
    rx=average_ranks(x);ry=average_ranks(y);observed=pearson(rx,ry);rng=random.Random(seed)
    extreme=0;permuted=list(ry)
    for _ in range(permutations):
        rng.shuffle(permuted)
        if abs(pearson(rx,permuted))>=abs(observed)-1e-15:extreme+=1
    return observed,(extreme+1)/(permutations+1)

# H1-H2: direction of change across UFs, Jan-Oct 2024 vs Jan-Oct 2025.
tests=[]
for metric,field,hypothesis in [
    ('SIA','sia_variacao_percentual','H1: direção da mudança SIA entre UFs (jan-out)'),
    ('SIH_N17_N19','sih_variacao_percentual','H2: direção da mudança SIH entre UFs (jan-out)')]:
    result=sign_test([float(r[field]) for r in cmp_uf])
    tests.append({'hipotese':hypothesis,'metodo':'teste binomial exato de sinais','n_ufs':result['n'],
                  'efeito':f"{result['positivos']} aumentos; {result['negativos']} reduções; mediana da variação={result['mediana_diferenca']:.2f}%",
                  'estatistica':result['positivos'],'p_bruto':result['p_bruto']})

# H3: ecological association of Jan-Oct quantities per population within year.
rate_rows=[]
for year,seed in [(2024,2026110501),(2025,2026110502)]:
    for uf in sorted({r['sigla_uf'] for r in panel_rows}):
        rows=[uf_rows[(uf,year,m)] for m in period_months]
        assert len(rows)==10
        pop=int(rows[0]['populacao_ibge_referencia_anual'])
        sia_total=sum(int(r['sia_hemodialise_quantidade_aprovada']) for r in rows)
        sih_total=sum(int(r['sih_n17_n19_internacoes_tabnet']) for r in rows)
        rate_rows.append({'cod_uf':rows[0]['cod_uf'],'sigla_uf':uf,'regiao':rows[0]['regiao'],'ano':year,
                          'periodo':'janeiro-outubro','sia_qtd_aprovada_jan_out':sia_total,
                          'sih_n17_n19_jan_out':sih_total,'populacao_ibge_1_julho':pop,
                          'sia_quantidade_por_100mil':round(sia_total/pop*100000,3),
                          'sih_internacoes_por_100mil':round(sih_total/pop*100000,3)})
    subset=[r for r in rate_rows if r['ano']==year]
    rho,p=spearman_permutation([r['sia_quantidade_por_100mil'] for r in subset],
                               [r['sih_internacoes_por_100mil'] for r in subset],seed)
    tests.append({'hipotese':f'H3: SIA por população vs SIH por população ({year}, jan-out)',
                  'metodo':'Spearman; permutação Monte Carlo 9.999','n_ufs':len(subset),
                  'efeito':f'rho={rho:.3f}','estatistica':rho,'p_bruto':p})
write_csv(out_dir/'associacoes_ecologicas_uf_ano.csv',rate_rows)

# H4: association between Jan-Oct percentage changes across UFs.
rho,p=spearman_permutation([float(r['sia_variacao_percentual']) for r in cmp_uf],
                           [float(r['sih_variacao_percentual']) for r in cmp_uf],2026110503)
tests.append({'hipotese':'H4: mudança percentual SIA vs SIH entre UFs (jan-out)',
              'metodo':'Spearman; permutação Monte Carlo 9.999','n_ufs':27,
              'efeito':f'rho={rho:.3f}','estatistica':rho,'p_bruto':p})
order=sorted(range(len(tests)),key=lambda i:tests[i]['p_bruto']);last=0.0
for rank,idx in enumerate(order):
    adj=min(1.0,(len(tests)-rank)*tests[idx]['p_bruto']);last=max(last,adj)
    tests[idx]['p_holm']=format(last,'.8g');tests[idx]['p_bruto']=format(tests[idx]['p_bruto'],'.8g')
    tests[idx]['estatistica']=round(float(tests[idx]['estatistica']),4)
write_csv(out_dir/'testes_hipoteses_exploratorios.csv',tests)
print('Hipóteses principais usam janeiro-outubro, janela anterior às quedas concentradas em nov/dez.')
for row in tests:print(row['hipotese'],'|',row['efeito'],'| p=',row['p_bruto'],'| p-Holm=',row['p_holm'])


Hipóteses principais usam janeiro-outubro, janela anterior às quedas concentradas em nov/dez.
H1: direção da mudança SIA entre UFs (jan-out) | 25 aumentos; 2 reduções; mediana da variação=3.30% | p= 5.6475401e-06 | p-Holm= 2.82377e-05
H2: direção da mudança SIH entre UFs (jan-out) | 17 aumentos; 10 reduções; mediana da variação=2.38% | p= 0.24778856 | p-Holm= 0.6336
H3: SIA por população vs SIH por população (2024, jan-out) | rho=-0.090 | p= 0.6553 | p-Holm= 0.6553
H3: SIA por população vs SIH por população (2025, jan-out) | rho=-0.288 | p= 0.1451 | p-Holm= 0.5804
H4: mudança percentual SIA vs SIH entre UFs (jan-out) | rho=-0.247 | p= 0.2112 | p-Holm= 0.6336


**Leitura dos testes:** p-valores não medem importância clínica nem removem confundimento. Unidades federativas podem compartilhar tendências, políticas e estrutura de serviços; a independência assumida pelos testes é aproximada. Os indicadores usam contagens administrativas agregadas, população aproximada de referência anual e somente 27 unidades. Interpretar resultados como sinais para investigação e planejamento, não como efeitos do tratamento.


Para explorar diferenças por mês sem comparar meses distintos, comparamos cada mês de 2025 ao mesmo mês de 2024, pareando as UFs. O teste binomial de sinais avalia se aumentos e reduções estão equilibrados entre UFs; Holm é aplicado separadamente aos meses SIA e SIH. Dezembro SIH é mostrado como controle de qualidade, mas excluído do teste: em 2025 soma 6.502 em 26 UFs, contra 12.647 nas mesmas UFs em 2024 (−48,59%), além de RR ausente. A magnitude sugere possível incompletude/maturidade; confirmar antes de interpretar. Com apenas dois anos, os resultados não estabelecem sazonalidade estável ou tendência de longo prazo.


In [7]:
# Testes mensais pareados por UF; nenhum dado de origem é alterado.
monthly_tests = []
for metric, field in [('SIA','sia'),('SIH_N17_N19','sih')]:
    for month in range(1,13):
        diffs=[]
        pairs=[]
        for row in cmp_uf:
            match24=next(r for r in panel_rows if r['sigla_uf']==row['sigla_uf'] and int(r['ano'])==2024 and int(r['mes'])==month)
            match25=next(r for r in panel_rows if r['sigla_uf']==row['sigla_uf'] and int(r['ano'])==2025 and int(r['mes'])==month)
            col='sia_hemodialise_quantidade_aprovada' if metric=='SIA' else 'sih_n17_n19_internacoes_tabnet'
            a=match24[col]; b=match25[col]
            if a in ('',None) or b in ('',None):
                continue
            pairs.append((int(a),int(b)))
            diffs.append(int(b)-int(a))
        st=sign_test(diffs)
        row_out={'familia':metric,'indicador':metric,'mes':month,'n_ufs':st['n'],
                 'aumentos':st['positivos'],'reducoes':st['negativos'],'empates':st['empates'],
                 'mediana_diferenca_2025_menos_2024':st['mediana_diferenca'],
                 'soma_2024':sum(a for a,b in pairs),'soma_2025':sum(b for a,b in pairs),
                 'status':'avaliado','motivo_exclusao':'','p_bruto':st['p_bruto'],'p_holm':''}
        # A queda abrupta em dezembro SIH e a ausência de RR sinalizam possível
        # incompletude da competência; conservar descrição, mas não testar inferencialmente.
        if metric=='SIH_N17_N19' and month in (11,12):
            row_out['status']='excluido_provisorio_qualidade'
            if month==11:
                row_out['motivo_exclusao']='Nov/2025 tem 10.545 registros em 27 UFs versus 13.054 em nov/2024 (-19,22%); 25 de 27 UFs caíram. Verificar maturidade/revisões antes de inferir.'
            else:
                row_out['motivo_exclusao']='Dez/2025 tem 6.502 registros em 26 UFs versus 12.647 nas mesmas 26 UFs em dez/2024 (-48,59%); verificar maturidade/revisões antes de inferir.'
            row_out['p_bruto']=''
            row_out['p_holm']=''
        monthly_tests.append(row_out)

for metric in ('SIA','SIH_N17_N19'):
    idxs=[i for i,r in enumerate(monthly_tests) if r['familia']==metric and r['status']=='avaliado']
    order=sorted(idxs,key=lambda i:monthly_tests[i]['p_bruto'])
    last=0.0
    for rank,i in enumerate(order):
        adj=min(1.0,(len(order)-rank)*monthly_tests[i]['p_bruto'])
        last=max(last,adj)
        monthly_tests[i]['p_bruto']=format(monthly_tests[i]['p_bruto'],'.8g')
        monthly_tests[i]['p_holm']=format(last,'.8g')
write_csv(out_dir/'testes_mensais_direcao.csv',monthly_tests)
print('Testes mensais por indicador; Holm dentro dos meses avaliados de cada indicador:')
for r in monthly_tests:
    if r['status']=='avaliado' and float(r['p_holm']) < .05:
        print(r['indicador'],r['mes'],'aumentos=',r['aumentos'],'reducoes=',r['reducoes'],
              'p-Holm=',r['p_holm'])
excluded=[r for r in monthly_tests if r['status']!='avaliado']
print('Testes avaliados:',sum(r['status']=='avaliado' for r in monthly_tests),
      '| excluídos provisoriamente:',len(excluded))
for r in excluded:
    print(r['indicador'],r['mes'],r['motivo_exclusao'])


Testes mensais por indicador; Holm dentro dos meses avaliados de cada indicador:
SIA 1 aumentos= 24 reducoes= 3 p-Holm= 0.00044310093
SIA 3 aumentos= 25 reducoes= 2 p-Holm= 5.6475401e-05
SIA 4 aumentos= 24 reducoes= 3 p-Holm= 0.00044310093
SIA 5 aumentos= 24 reducoes= 3 p-Holm= 0.00044310093
SIA 6 aumentos= 24 reducoes= 3 p-Holm= 0.00044310093
SIA 7 aumentos= 26 reducoes= 1 p-Holm= 5.0067902e-06
SIA 9 aumentos= 26 reducoes= 1 p-Holm= 5.0067902e-06
SIA 10 aumentos= 24 reducoes= 3 p-Holm= 0.00044310093
SIA 12 aumentos= 24 reducoes= 3 p-Holm= 0.00044310093
Testes avaliados: 22 | excluídos provisoriamente: 2
SIH_N17_N19 11 Nov/2025 tem 10.545 registros em 27 UFs versus 13.054 em nov/2024 (-19,22%); 25 de 27 UFs caíram. Verificar maturidade/revisões antes de inferir.
SIH_N17_N19 12 Dez/2025 tem 6.502 registros em 26 UFs versus 12.647 nas mesmas 26 UFs em dez/2024 (-48,59%); verificar maturidade/revisões antes de inferir.


O arquivo `testes_mensais_direcao.csv` contém diferenças pareadas por UF, frequências direcionais, p-valores e status de inclusão. Dezembro SIH é sinalizado para investigação de qualidade e não recebe teste inferencial. Interpretar diferenças com cobertura e contexto; não inferir mecanismo clínico ou causalidade.
